# Part 1 — Data Acquisition, Cleaning, and Exploratory Analysis

**Dataset:** Synthetic online retail order-line data (`raw_online_retail.csv`), generated for this
assignment. Each row represents one product line within a customer order: who bought it
(segment, region, age), what was bought (category, quantity, unit price, discount), how much
it cost (revenue, shipping), and how the order went (rating, delivery days).

The dataset was generated with realistic messiness baked in on purpose: missing values at
different rates per column, duplicate rows, a numeric column stored as text, skewed
distributions, and genuine outliers — so every cleaning/EDA technique below has something
real to catch.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style="whitegrid")
os.makedirs("plots", exist_ok=True)
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 20)


## Task 1 — Load the dataset

In [ ]:
df = pd.read_csv("raw_online_retail.csv")
print("Shape:", df.shape)
df.head()


In [ ]:
df.dtypes


## Task 2 — Null value analysis

Compute the count and percentage of missing values per column, flag any column above the
20% threshold, and median-fill the numeric columns that fall below it.

**Why median, not mean, for filling?** Several numeric columns in this dataset (`revenue`,
`quantity`, `delivery_days`) are heavily right-skewed — a handful of very large orders or
very late deliveries pull the mean upward, so the mean no longer represents a "typical" row.
The median is robust to that pull, so filling nulls with it doesn't introduce values that are
systematically too high.


In [ ]:
null_counts = df.isnull().sum()
null_pct = (df.isnull().sum() / df.shape[0]) * 100
null_table = pd.DataFrame({"null_count": null_counts, "null_pct": null_pct.round(2)})
null_table = null_table.sort_values("null_pct", ascending=False)
null_table


In [ ]:
high_null_cols = null_table[null_table["null_pct"] > 20].index.tolist()
print("Columns exceeding 20% null rate:", high_null_cols)

low_null_numeric = [
    c for c in df.select_dtypes(include=np.number).columns
    if c not in high_null_cols and df[c].isnull().sum() > 0
]
print("Numeric columns below 20% nulls -> median-filled:", low_null_numeric)

for col in low_null_numeric:
    df[col] = df[col].fillna(df[col].median())

df.isnull().sum()


**Finding:** `shipping_cost` is the only column above the 20% threshold (~29% missing).
It is *not* filled at this stage — filling nearly a third of a column with a single statistic
would distort its distribution — it's left for a deliberate strategy in Part 2 (e.g. flag
as "missing" or model-based imputation) and only reported here.


## Task 3 — Duplicate detection and removal

Exact duplicate rows are almost certainly a data-pipeline artifact (e.g. an order line
re-exported twice) rather than genuine repeat data, so they are dropped outright.


In [ ]:
n_dupes = df.duplicated().sum()
print("Duplicate rows found:", n_dupes)

null_pct_before_dedup = (df.isnull().sum() / df.shape[0] * 100).round(3)
df = df.drop_duplicates()
null_pct_after_dedup = (df.isnull().sum() / df.shape[0] * 100).round(3)

print("Rows removed:", n_dupes)
print("New shape:", df.shape)

pd.DataFrame({"null_pct_before": null_pct_before_dedup, "null_pct_after": null_pct_after_dedup})


**Finding:** 40 duplicate rows were removed. The null percentages barely move
(e.g. `shipping_cost` 29.118% → 29.1%) — the duplicated rows happened to carry a
representative mix of nulls, so removing them didn't skew the missingness picture.


## Task 4 — Data type correction

`unit_price` was loaded as text because the raw export mixed a `$` currency prefix with a
few literal `"N/A"` placeholders — pandas can't infer a numeric dtype from that, so it fell
back to `object`. It's converted with `pd.to_numeric(..., errors="coerce")` after stripping
the `$` sign. The three repetitive category-style text columns (`product_category`,
`customer_segment`, `region`) are converted to pandas `category` dtype, which stores each
value once in a lookup table instead of repeating the string on every row.


In [ ]:
mem_before = df.memory_usage(deep=True).sum()
print("Memory usage BEFORE conversion (bytes):", mem_before)

df["unit_price"] = df["unit_price"].replace("N/A", np.nan)
df["unit_price"] = df["unit_price"].str.replace("$", "", regex=False)
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

up_null_pct = df["unit_price"].isnull().mean() * 100
print(f"unit_price null % after coercion: {up_null_pct:.2f}%")
if up_null_pct <= 20:
    df["unit_price"] = df["unit_price"].fillna(df["unit_price"].median())

for col in ["product_category", "customer_segment", "region"]:
    df[col] = df[col].astype("category")

mem_after = df.memory_usage(deep=True).sum()
print("Memory usage AFTER conversion (bytes):", mem_after)
print(f"Memory reduction: {mem_before - mem_after} bytes "
      f"({(mem_before - mem_after) / mem_before * 100:.2f}%)")
df.dtypes


**Finding:** Memory usage dropped from 734,807 bytes to 302,684 bytes — a **58.8%
reduction** — almost entirely from converting the three repetitive string columns to
`category`, plus `unit_price` becoming a compact `float64` instead of a Python object column.


## Task 5 — Descriptive statistics and skewness


In [ ]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != "order_id"]
df[numeric_cols].describe()


In [ ]:
skew_vals = df[numeric_cols].skew().sort_values(key=lambda s: s.abs(), ascending=False)
skew_vals


In [ ]:
most_skewed_col = skew_vals.index[0]
second_skewed_col = skew_vals.index[1]
print(f"Most skewed column: {most_skewed_col} (skew={skew_vals.iloc[0]:.3f})")
print(f"Second most skewed column: {second_skewed_col} (skew={skew_vals.iloc[1]:.3f})")


**Finding:** `quantity` has the highest absolute skew (**~10.0**), followed closely by
`revenue` (**~8.0**). Both are strongly **positively skewed**: most order lines have a small,
tightly-clustered quantity (1–5 units) and modest revenue, but a small number of bulk orders
(quantity in the 25–50 range) and a handful of very large-ticket orders stretch a long tail
out to the right.

**Consequence for mean-imputation:** because the mean gets dragged toward that long right
tail, filling missing `quantity` or `revenue` values with the column mean would insert values
noticeably larger than what a "typical" order actually looks like — which is exactly why the
median (Task 2, Task 8a) is the safer choice for these two columns.


## Task 6 — Outlier detection with IQR


In [ ]:
iqr_cols = ["revenue", "quantity", "delivery_days"]
iqr_report = {}
for col in iqr_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((df[col] < lower) | (df[col] > upper)).sum()
    iqr_report[col] = dict(Q1=Q1, Q3=Q3, IQR=IQR, lower=lower, upper=upper, n_outliers=n_out)
    print(f"{col}: Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}, "
          f"bounds=[{lower:.2f}, {upper:.2f}], outliers={n_out}")


**Finding & handling decision:**
- `revenue`: 149 rows fall outside `[-653.75, 1230.03]` — effectively all on the high side
  (large multi-item or high-ticket orders). These look like *genuine* big orders, not data
  errors.
- `quantity`: 10 rows exceed the upper bound of 7 — these are the deliberate bulk-order rows
  (25–50 units).
- `delivery_days`: 27 rows exceed the upper bound of 8 — plausible real shipping delays.

None of these are dropped. All three look like real business events rather than data-entry
mistakes, and dropping them would throw away the exact "big order" / "late delivery" signal
a downstream model would want to learn from. **Decision for Part 2:** retain all three, but
apply a **cap (winsorize) on `revenue` and `quantity`** at the upper IQR bound before feeding
a linear model (since a few extreme rows can dominate a squared-error loss), while leaving
`delivery_days` untouched for a tree-based model, which is not sensitive to that kind of
outlier.


## Task 7 — Visualizations

### 7.1 Line plot — Revenue over time

In [ ]:
df_sorted = df.sort_values("order_date")
plt.figure(figsize=(10, 5))
plt.plot(df_sorted["order_date"], df_sorted["revenue"], linewidth=0.6, alpha=0.8)
plt.title("Revenue Over Time (per order line)")
plt.xlabel("Order Date")
plt.ylabel("Revenue ($)")
plt.tight_layout()
plt.savefig("plots/01_line_revenue_over_time.png", dpi=110)
plt.show()


Revenue per order line is noisy with no strong upward/downward trend across the year — spikes correspond to the large-ticket orders identified as outliers in Task 6.

### 7.2 Bar chart — Mean revenue by product category

In [ ]:
plt.figure(figsize=(8, 5))
cat_means = df.groupby("product_category", observed=True)["revenue"].mean().sort_values(ascending=False)
plt.bar(cat_means.index, cat_means.values, color="#4C72B0")
plt.title("Mean Revenue by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Mean Revenue ($)")
plt.tight_layout()
plt.savefig("plots/02_bar_mean_revenue_by_category.png", dpi=110)
plt.show()


Furniture and Electronics carry the highest mean revenue per line (higher unit prices), while Office Supplies is lowest, consistent with how the base prices were structured.

### 7.3 Histogram — Most skewed column (`quantity`)

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df[most_skewed_col], bins=20, kde=True, color="#DD8452")
plt.title(f"Distribution of {most_skewed_col} (skew={skew_vals.iloc[0]:.2f})")
plt.xlabel(most_skewed_col)
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("plots/03_histogram_most_skewed.png", dpi=110)
plt.show()


**Shape description:** the vast majority of rows sit between 1 and 5 units — a tall,
tight cluster near zero — with a thin, long tail of bulk orders (25–50 units) stretching far
to the right. This is a textbook strongly right-skewed ("positively skewed") distribution.


### 7.4 Scatter plot — Quantity vs Revenue

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="quantity", y="revenue", alpha=0.5, hue="product_category")
plt.title("Quantity vs Revenue")
plt.xlabel("Quantity")
plt.ylabel("Revenue ($)")
plt.tight_layout()
plt.savefig("plots/04_scatter_quantity_vs_revenue.png", dpi=110)
plt.show()


**Interpretation:** there is a visible, but only moderate, positive relationship — as
quantity rises, revenue tends to rise too (Pearson r ≈ 0.12 on raw values, rising to
Spearman ≈ 0.33; see Task 8b), but revenue is driven at least as much by `unit_price`, so the
same quantity can land at very different revenue levels depending on product category
(reflected in the color spread).


### 7.5 Box plot — Revenue by region

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="region", y="revenue")
plt.title("Revenue Distribution by Region")
plt.xlabel("Region")
plt.ylabel("Revenue ($)")
plt.tight_layout()
plt.savefig("plots/05_boxplot_revenue_by_region.png", dpi=110)
plt.show()


**Interpretation:** median revenue is very similar across all five regions (no region
is a clear high- or low-revenue market), but spread differs — North shows the widest box and
the most extreme high-end outliers, meaning revenue is far less predictable there than in,
say, Central. This matches the grouped standard deviations computed in Task 8c.


### 7.6 Correlation heat map

In [ ]:
plt.figure(figsize=(9, 7))
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Heat Map (Pearson)")
plt.tight_layout()
plt.savefig("plots/06_correlation_heatmap.png", dpi=110)
plt.show()


In [ ]:
upper_mask = np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
corr_unstacked = corr_matrix.where(upper_mask).unstack().dropna()
top_pair = corr_unstacked.abs().sort_values(ascending=False).index[0]
top_pair_val = corr_matrix.loc[top_pair[0], top_pair[1]]
print(f"Highest |correlation| pair: {top_pair} = {top_pair_val:.3f}")


**Interpretation:** `unit_price` and `revenue` show the strongest correlation (r ≈ 0.65),
which is expected almost by construction: `revenue = quantity × unit_price × (1 - discount)`,
so `unit_price` is a direct mathematical input to `revenue`, not just an associated variable.
This is a case where the correlation **is** substantially causal/mechanical rather than
coincidental — but if this were observed in a dataset where revenue's formula wasn't known,
a plausible alternative explanation would be a **confounding "product tier" variable**:
premium products tend to have both a higher unit price *and* independently higher demand
or larger typical order sizes, which could inflate the observed correlation beyond the
direct price → revenue arithmetic.


## Task 8a — Imputation strategy comparison (mean vs. median)

In [ ]:
df_raw = pd.read_csv("raw_online_retail.csv")
df_raw["unit_price"] = pd.to_numeric(
    df_raw["unit_price"].replace("N/A", np.nan).str.replace("$", "", regex=False),
    errors="coerce"
)

target_cols = [most_skewed_col, second_skewed_col]
for col in target_cols:
    src = df_raw[col] if col in df_raw.columns else df[col]
    print(f"{col}: mean={src.mean():.3f}, median={src.median():.3f}")


In [ ]:
for col in target_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

df[target_cols].isnull().sum()


**Choice & justification:** for both `quantity` (mean 3.13 vs. median 3.00) and
`revenue` (mean ≈ 432.11 vs. median ≈ 166.31) the mean sits well above the median — the
signature of positive skew pulling the mean upward via the bulk-order / high-ticket tail.
The **median** is used for imputing both, since it represents the typical order far better
than a mean that's been inflated by a small number of large orders.


## Task 8b — Spearman rank correlation vs. Pearson


In [ ]:
pearson_matrix = df[numeric_cols].corr(method="pearson")
spearman_matrix = df[numeric_cols].corr(method="spearman")
pearson_matrix.round(3)


In [ ]:
spearman_matrix.round(3)


In [ ]:
diff_matrix = (spearman_matrix - pearson_matrix).abs()
upper_mask2 = np.triu(np.ones(diff_matrix.shape), k=1).astype(bool)
diff_unstacked = diff_matrix.where(upper_mask2).unstack().dropna()
top3_pairs = diff_unstacked.sort_values(ascending=False).head(3)

rows = []
for (a, b), diff in top3_pairs.items():
    rows.append({"pair": f"{a} vs {b}", "pearson": round(pearson_matrix.loc[a, b], 3),
                 "spearman": round(spearman_matrix.loc[a, b], 3), "abs_diff": round(diff, 3)})
diff_table = pd.DataFrame(rows)
diff_table


**Interpretation of the three largest gaps:**
1. **`revenue` vs `unit_price`** (Pearson 0.650, Spearman 0.883, diff 0.232): |Spearman| >
   |Pearson| — the relationship is **monotonic but non-linear**. This makes sense: `revenue`
   is a *product* of `unit_price` with `quantity` and `(1 - discount)`, so higher unit price
   reliably pushes revenue up in rank order, but not in a constant straight-line way.
2. **`revenue` vs `quantity`** (Pearson 0.120, Spearman 0.332, diff 0.212): |Spearman| >
   |Pearson| — again **monotonic but non-linear**. Pearson is muted here because a few
   extreme high-revenue, low-quantity rows (high-ticket single-item orders) and the bulk-order
   outliers work against a straight-line fit, while the *rank* relationship (more units →
   more revenue) is fairly consistent.
3. **`shipping_cost` vs `unit_price`** (Pearson 0.495, Spearman 0.388, diff 0.107): here
   |Pearson| ≥ |Spearman| — this one is **closer to a linear relationship** than the other
   two; the small drop in Spearman suggests a modest departure from strict monotonicity
   rather than a strongly non-linear pattern.

**Which measure to use for Part 2 feature selection:** **Spearman**, for the revenue-related
pairs. Since the target-adjacent relationships (`unit_price`→`revenue`, `quantity`→`revenue`)
are monotonic but not linear, Pearson would *understate* how useful these features are for a
non-linear/tree-based model. Spearman gives a fairer signal of feature relevance in that
setting, and is also more robust to the outliers documented in Task 6.


## Task 8c — Grouped aggregation (`region` × `revenue`)


In [ ]:
group_col = "region"
agg_col = "revenue"
grouped = df.groupby(group_col, observed=True)[agg_col].agg(["mean", "std", "count"])
grouped


In [ ]:
highest_mean_group = grouped["mean"].idxmax()
highest_std_group = grouped["std"].idxmax()
mean_ratio = grouped["mean"].max() / grouped["mean"].min()
print(f"Highest mean group: {highest_mean_group} ({grouped['mean'].max():.2f})")
print(f"Highest std group: {highest_std_group} ({grouped['std'].max():.2f})")
print(f"Ratio highest-mean/lowest-mean: {mean_ratio:.3f}")


**Findings:**
- **Highest mean revenue:** South (\$430.72). **Highest std dev:** North (\$902.53).
- **Is high within-group std a concern?** Yes. North's standard deviation (≈\$903) is more
  than double its own mean (≈\$423) — knowing an order came from North tells you almost
  nothing about what its revenue will be, since outcomes swing enormously within that single
  group. A model that only knew `region = North` would still face huge uncertainty on revenue.
- **Mean ratio (highest/lowest):** 1.064 — the group means are within about 6% of each other.
  A ratio this close to 1 indicates `region` carries **very little predictive signal** for
  revenue on its own; the real explanatory power lives in `unit_price`/`quantity`/`product_category`,
  not which region the order shipped to.


## Task 9 — Save the cleaned dataset

In [ ]:
df.to_csv("cleaned_data.csv", index=False)
print("Saved cleaned_data.csv, shape:", df.shape)
df.isnull().sum()


`shipping_cost` still shows nulls in the saved file — this is intentional (Task 2): a
column above the 20% missing threshold is documented, not blindly filled, and its treatment
is deferred to Part 2.
